# Демо: контекстные менеджеры

Прокликай Shift+Enter каждую ячейку и посмотри, как `with` гарантирует освобождение ресурса даже при исключении, чем он лучше `try/finally`, и как написать собственный контекстный менеджер через `@contextmanager`-декоратор. В конце — три мини-задания.

## Часть 1. Зачем нужен `with` — задача гарантированной очистки

Сейчас посмотрим, что происходит без `with`. Открыли файл, считали содержимое, забыли закрыть — файловый дескриптор остался у процесса. На одном файле незаметно; на тысяче запросов в секунду — `OSError: too many open files`.

In [1]:
import tempfile
import os

# Создадим временный файл с парой строк
with tempfile.NamedTemporaryFile("w", delete=False, suffix=".txt") as tmp:
    tmp.write("первая строка\n")
    tmp.write("вторая строка\n")
    tmp_path = tmp.name

print(f"файл создан: {tmp_path}")

файл создан: /var/folders/gf/l4lkvdqx599_bjs2yk5_s4d40000gn/T/tmp8pa51nsi.txt


Наивный вариант — открыть, прочитать, закрыть. Если между `open` и `close` вылетит исключение, `close` никто не позовёт.

In [2]:
f = open(tmp_path)
content = f.read()
print(content)
f.close()    # надо помнить вручную

первая строка
вторая строка



Надёжная версия без `with` — через `try/finally`. Закрытие гарантировано и при успехе, и при исключении. Но шумно — лишние строки на каждый файл.

In [3]:
f = open(tmp_path)
try:
    content = f.read()
    print(content)
finally:
    f.close()    # выполнится даже если в try упало

первая строка
вторая строка



## Часть 2. Синтаксис `with` — тот же контракт, но без шума

`with expr as name:` — Python сам вызывает закрытие при выходе из блока, успешном или нет. Тот же контракт `try/finally`, спрятанный за коротким синтаксисом.

In [4]:
with open(tmp_path) as f:
    content = f.read()
    print(content)
# здесь файл уже закрыт — гарантированно

print(f"f.closed = {f.closed}")    # True

первая строка
вторая строка

f.closed = True


А что если попытаться прочитать файл **после** `with`? Файл уже закрыт, Python поднимет `ValueError`. Всё, что нужно из файла, делается внутри блока, а не после него.

In [5]:
try:
    f.read()
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: I/O operation on closed file.


## Часть 3. Исключение внутри блока — закрытие всё равно сработает

Главная гарантия `with`: даже если внутри блока вылетит исключение, выход из контекста всё равно произойдёт. Файл закроется, лок отпустится, соединение разорвётся.

In [6]:
try:
    with open(tmp_path) as f:
        content = f.read()
        raise RuntimeError("искусственное падение")
except RuntimeError as e:
    print(f"поймали: {e}")

# несмотря на исключение, файл закрыт
print(f"f.closed = {f.closed}")    # True

поймали: искусственное падение
f.closed = True


## Часть 4. `with` работает не только с файлами

Любой объект с методами `__enter__` / `__exit__` совместим с `with`. Стандартные примеры — блокировки потоков, HTTP-клиенты, пулы. Покажем на блокировке.

In [7]:
import threading

lock = threading.Lock()

with lock:
    print("внутри блока — лок захвачен")
    print(f"locked = {lock.locked()}")
# при выходе из блока лок автоматически отпускается
print(f"locked после блока = {lock.locked()}")

внутри блока — лок захвачен
locked = True
locked после блока = False


## Часть 5. `@contextmanager` — свой менеджер из обычной функции

Класс с `__enter__` / `__exit__` мы напишем на следующей неделе, когда разберём магические методы. А сейчас короткий путь: декоратор `contextlib.contextmanager` превращает генератор с одним `yield` в полноценный контекстный менеджер.

Шаблон: код **до** `yield` — это `__enter__` (вход в контекст). Код **после** `yield` — это `__exit__` (очистка).

In [8]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label):
    start = time.perf_counter()
    print(f"[{label}] старт")
    yield                        # передача управления внутрь with-блока
    elapsed = time.perf_counter() - start
    print(f"[{label}] финиш за {elapsed:.4f} сек")

with timer("подсчёт суммы"):
    total = sum(range(1_000_000))
print(f"total = {total}")

[подсчёт суммы] старт
[подсчёт суммы] финиш за 0.0080 сек
total = 499999500000


А что если внутри блока вылетит исключение? Без `try/finally` код после `yield` не выполнится — очистки не будет. Поэтому в реальном `@contextmanager` пишут `try/yield/finally`:

In [9]:
@contextmanager
def safe_timer(label):
    start = time.perf_counter()
    print(f"[{label}] старт")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] финиш за {elapsed:.4f} сек (даже при исключении)")

try:
    with safe_timer("вычисление"):
        x = sum(range(100_000))
        raise RuntimeError("упало")
except RuntimeError as e:
    print(f"поймали: {e}")

[вычисление] старт
[вычисление] финиш за 0.0008 сек (даже при исключении)
поймали: упало


## Часть 6. Сравнение: `try/finally` vs `with`

Тот же контракт «гарантировать очистку» можно записать тремя способами. Покажем все три на одном примере — обработка содержимого файла.

In [10]:
# Вариант 1 — try/finally вручную
f = open(tmp_path)
try:
    data1 = f.read().upper()
finally:
    f.close()

# Вариант 2 — with со встроенным менеджером (стандарт для файлов)
with open(tmp_path) as f:
    data2 = f.read().upper()

# Вариант 3 — собственный @contextmanager (если стандартного нет)
@contextmanager
def opened(path):
    fh = open(path)
    try:
        yield fh
    finally:
        fh.close()

with opened(tmp_path) as f:
    data3 = f.read().upper()

print(data1 == data2 == data3)    # True — результат одинаковый
print(data2)

True
ПЕРВАЯ СТРОКА
ВТОРАЯ СТРОКА



Уберём временный файл за собой:

In [11]:
os.unlink(tmp_path)
print(f"удалили {tmp_path}")

удалили /var/folders/gf/l4lkvdqx599_bjs2yk5_s4d40000gn/T/tmp8pa51nsi.txt


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Перепиши блок чтения файла так, чтобы использовать `with` вместо ручного `open` / `close`. Используй переменную `tmp_path` (надо сначала создать тестовый файл — есть готовая ячейка).

**Задание 2.** Напиши контекстный менеджер `working_directory(path)` через `@contextmanager`. На входе — запоминает текущий каталог и переходит в `path`; на выходе — возвращается обратно. Подсказка: `os.getcwd()` и `os.chdir(path)`. Не забудь `try/finally`.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [12]:
# Задание 1
import tempfile, os
with tempfile.NamedTemporaryFile("w", delete=False, suffix=".txt") as tmp:
    tmp.write("задание 1")
    task1_path = tmp.name

# Перепиши блок ниже на with-стиль:
# f = open(task1_path)
# data = f.read()
# print(data)
# f.close()

# Твой код:

os.unlink(task1_path)


In [13]:
# Задание 2
# from contextlib import contextmanager
# import os
#
# @contextmanager
# def working_directory(path):
#     ...

# Проверка:
# print(os.getcwd())
# with working_directory('/tmp'):
#     print(os.getcwd())   # /tmp
# print(os.getcwd())       # снова исходный


In [14]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
from contextlib import contextmanager

@contextmanager
def trace(name):
    print(f'enter {name}')
    yield name
    print(f'exit  {name}')

with trace('A') as a, trace('B') as b:
    print(f'inside: {a}, {b}')

# Какой будет порядок строк? ?


enter A
enter B
inside: A, B
exit  B
exit  A
